# Mini-Project: Bespoke Retrieval-Augmented Generation (RAG)


## Overview


In this project, you will build a hybrid system that combines three components into a single RAG-style pipeline:

- A **language model (GPT-2)** for text generation.
- A **Query-Key Memory Network (QK-MemNet)** for factual retrieval.
- A **router** that decides whether to use autoregressive generation or to invoke the retrieval model.

This assignment combines language modeling, retrieval, text classification, and data curation.

The overall goal is to build a system that generates biographies of real people. At each step of generation, a router model examines the current text that has been generated so far and decides: "Should we **generate** the next token with GPT-2, or **retrieve** a fact from structured memory?"

**Note:** This system is **not expected to produce perfect biographies**. GPT-2 is a small model and the QK-MemNet uses bag-of-words representations. The goal is to demonstrate the *architecture*, decision-making between retrieval and generation, and integration of symbolic memory with neural models.

## Learning objectives

The learning objective of this assignment is to provide practical experience handling structured and unstructured data, training classifiers on real-world data, and incorporating different models into a single, practical generative system.

We ask you to envision that you have been tasked with creating a specialized system that generates biographies for famous people that tamps down on hallucinations as much as possible. With the very powerful LLMs available today, retrieval-augmented generation (RAG) might be achieved by using web search to pull some web pages for a person and then concatenating the contents of the web page along with a prompt, and letting a very large LLM do its thing with attention. However, this is not very exciting as an assignment that combines skills across the entire class.

Instead, we are going to pretend that we have an LLM that has not been able to pre-train on biographies of famous people (if you want, you can pretend that we are working on a system that generates biographies of un-famous people). To simulate this setting, we will work with GPT-2 because it is less capable of memorizing facts. This also means we can work with language models that are small enough to not need a lot of computing resources. No one will realistically use GPT-2, but GPT-2 has not memorized a lot of biographical information, so it suits our purposes.

We will also need a retriever. For this assignment, we will build a Query-Key Memory Network (QK-MemNet), inspired by Key-Value Memory Networks (KV-MemNets). A Memory Network learns to index into a data source (external memory) using attention mechanisms. They were considered a promising pathway to more sophisticated retrieval and generation capabilities before the Transformer came along. A Query-Key Memory Network assumes that memory is structured as a hash table and learns the mapping from an arbitrary query to the appropriate keys in order to retrieve its associated values. Where a hash table requires an exact match between query and key, a QK-MemNet learns a function that maps queries to keys. Memory networks use attention in a way that somewhat corresponds to Transformers. This makes sense because Transformers use their K, V, and Q sub-networks that are inspired by hash tables. We thus get to play with something that shares a lot of commonalities with Transformers without implementing all the complexity of the Transformer (and for which the code is already given earlier in the lectures).

The QK-MemNet will store biographical facts (birthdates, birth places, etc.), but not full biographies. So we will have GPT-2 generate biographies, and it will likely hallucinate the facts. How do we know when to retrieve? For this, we will train a classifier that will tell us whether the next token(s) should be retrieved or if we should let GPT-2 generate the next tokens. Thus the generation flow looks like this:

```
GPT-2 generates token → Classifier checks: is a fact needed next?
                            ↳ YES → QK-MemNet retrieves fact → inject into context
                            ↳ NO  → continue generating
```

The router will be a fine-tuned version of BERT, trained to predict whether the next token(s) should be facts. To fine-tune BERT, we will need to construct a training set of sentence fragments, some of which are followed by facts and some of which are not. Thus, we will need to prepare a specialized dataset. We will provide you with the raw materials to do this.

There are other ways this RAG project might have been done. We might have let the LM generate and then tried to detect hallucinated facts and "rewind" and invoke the retriever. That has its own challenges. We might fine-tune the LM to generate placeholder text and then grab those placeholders and call the retriever when a placeholder appears. We have chosen to insist on a particular structure for the assignment that involves creating datasets, training a classifier, and building a bespoke model architecture from scratch for their educational value.

By the end of this project, you will have practiced:

1. Using open-weight language models.
2. Creating supervised learning datasets that align structured knowledge with unstructured text.
3. Training a neural binary classifier off annotated data.
4. Implementing attention-based neural architectures for retrieval.
5. Building specialized RAG systems.

## How to use this Notebook


- Cells marked with `### START CODE HERE ###` and `### END CODE HERE ###` are where you write your solutions.
- Do **not** modify provided helper functions or test cells unless instructed.
- Run cells **in order**, later parts depend on earlier results.

The project is split across three notebooks:

- **Part A: Router**
- **Part B: Memory Network**
- **Parts C–D: GPT-2 fine-tuning (optional) and RAG integration**

This notebook covers Part A. Save your annotation datasets for use in Part B.

You will likely spend the most time on Part A, so we suggest getting started early.


## Grading Summary: Part A

| Section | Title | Function(s) | Checks | Points |
| --- | --- | --- | --- | ---: |
| A.2.b | Build Router Dataset | `make_records_for_person`, `text_processing_function` | 1.1–1.5 | 9 |
| A.3.b | Preprocess Data | `tokenize_record` | 2.1–2.3 | 5 |
| A.3.d | Train the Router | `train_router` | 2.4, 3.1–3.2, 4.1–4.2 | 9 |
| **Total** | | | | **23** |

The checks listed above are the tests used by the autograder. You can try them using the Student Self-check at the end of this notebook.


# Setting Up

In [ ]:
!pip install unidecode --quiet

In [ ]:
import json
import re
import random
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional
from copy import deepcopy
import unidecode

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

The following cell mounts Google Drive to the Colab environment so that files stored in Drive can be accessed directly from the notebook. This is only necessary if you are using Google Colab.

After mounting, we define the root directory of the project located in Google Drive. This provides a centralized path for all file references in the project.

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/cs7650_final_project")  # adjust as needed

In [ ]:
MOUNT_GOOGLE_DRIVE = True

if MOUNT_GOOGLE_DRIVE:
  from google.colab import drive
  drive.mount("/content/drive", force_remount=False)

# Section A: Router (23 points)


This section builds a router model, which classifies a segment of text to determine if the next segment will contain a fact.

A prefix is a sub-string beginning with the start of a span. For example: the string ` "Lillian Gordy Carter was born in Richland" ` has the following prefixes:

1. Lillian
2. Lillian Gordy
3. Lillian Gordy Carter
4. Lillian Gordy Carter was
5. Lillian Gordy Carter was born
6. Lillian Gordy Carter was born in
7. Lillian Gordy Carter was born in Richland

Prefixes 1-5 and 7 are not followed by a fact to be retrieved from the knowledge base. Prefix 6 should be recognized as needing to be followed by a fact from the knowledge base.

**What you will build:** A classifier that examines a partial sentence (prefix) and predicts whether the next word is likely to be a factual entity that should be retrieved from a knowledge base.

## A.1. Load the provided Wikidata and SimpleWiki bios (0 points)


Data is pulled directly from a database of US politician biography data hosted on Hugging Face Hub (https://huggingface.co/datasets/specialsaucem/us-politicians).


Each record combines two kinds of data: WikiData structured facts about an individual and biographical introductory text, pulled from Simplified Wikipedia or normal Wikipedia, or both.

We will only be using records with a `simple-bio` entry, which contains simplified natural language.

People are split into train/dev/test with a proportional 80/10/10 split
(shuffled with a fixed random seed for reproducibility).



The following cells create 3 variables:
- `db_clean_train` / `db_clean_dev` / `db_clean_test`: structured Wikidata
  facts per person, keyed by lowercase full name -- e.g.
  `db_clean_train["calvin coolidge"]["date of birth"]`. Multiple values for
  the same attribute are joined with `"; "`.

A handful of people share the same lowercase name (e.g. two different
"Henry Clay"s) -- the first one encountered is kept and the rest are
dropped and printed below. Don't worry about that.


In [ ]:
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "specialsaucem/us-politicians"  # https://huggingface.co/datasets/specialsaucem/us-politicians

people_jsonl_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="people.jsonl",
    repo_type="dataset",
)

db_clean_all = {}
title_to_text_all = {}

with open(people_jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)

        simple_bio = rec.get("simple-bio")
        if not simple_bio:
            continue  # restrict to people with a Simple Wikipedia bio, same
                       # restriction the old full_properties_simple_bio.csv
                       # already applied.

        name_key = str(rec["name"]).strip().lower()

        if name_key in db_clean_all:
            continue

        db_clean_all[name_key] = rec["properties"]
        title_to_text_all[name_key] = simple_bio

print(f"\n{len(db_clean_all)} people total after deduplication.")

# Proportional 80/10/10 train/dev/test split, shuffled with a fixed seed
# (independent Random instance so this doesn't disturb the notebook's
# global `random` state, which was already seeded in the Setting Up section).
all_names = sorted(db_clean_all)
rng = random.Random(RANDOM_SEED)
rng.shuffle(all_names)

n_total = len(all_names)
n_train = round(n_total * 0.8)
n_dev = round(n_total * 0.1)

train_names = all_names[:n_train]
dev_names = all_names[n_train:n_train + n_dev]
test_names = all_names[n_train + n_dev:]

def _subset(d, names):
    return {n: d[n] for n in names}

db_clean_train = _subset(db_clean_all, train_names)
db_clean_dev = _subset(db_clean_all, dev_names)
db_clean_test = _subset(db_clean_all, test_names)

print(f"Split sizes: train={len(db_clean_train)}, dev={len(db_clean_dev)}, test={len(db_clean_test)}")


Print the split sizes and inspect one example from the train split.

In [ ]:
print(
    f"Split sizes: train={len(db_clean_train)}, dev={len(db_clean_dev)}, test={len(db_clean_test)}"
)

# Get the first biography name
example_name = 'lillian gordy carter'
print("One train example:")
print("Biography name:", example_name)
db_clean_all[example_name]


Build `title_to_text`, a lookup from each person's (lowercased) name to
their Simple Wikipedia biography text -- reusing the `simple-bio` values
already parsed from `people.jsonl` above, instead of downloading the old
Kaggle `simplenormal-wikipedia-sections` dataset. `people.jsonl` doesn't
carry a separate article title, so this is keyed the same way as
`db_clean_*` (lowercased full name).


In [ ]:
title_to_text = title_to_text_all
print("SimpleWiki bios loaded:", len(title_to_text))

Print one biography text example so you can see what the SimpleWiki input looks like.

In [ ]:
example_simplewiki_text = title_to_text[example_name]

print(f"SimpleWiki text example title: {example_name}")
print(example_simplewiki_text[:500])

## A.2. Create Router Training Labels (9 points)


In this part, the SimpleWiki biography is scanned sentence-by-sentence to find where Wikidata facts appear in the text. Each prefix ending just before a fact word becomes a *positive* training example (label=1), and each prefix ending before a non-fact word becomes a *negative* example (label=0).



### A.2.a. Load the spaCy tokenizer used for sentence scanning (0 points)

Use a lightweight English pipeline so the notebook can read token boundaries and POS tags for the label-generation logic.

In this step, we load a lightweight NLP tokenizer using spaCy. The SpaCy documentation is here: https://spacy.io/api

SpaCy is good for splitting text into word-level tokens, identifying the parts-of-speech for each token, identifying dependencies between tokens, splitting paragraphs into sentence, and identifying entities.

You do NOT need to modify this section, but you should understand how tokenization affects downstream tasks.


**Using spaCy**

The spaCy library does a reasonably good job at identifying the parts of speech for each word in a document. Why this is useful is because facts aren't just words associated with each person, but only occur as certain parts of speech, for example `NOUN`, `PROPN` (proper noun), and `NUM`. This information can help more accurately figure out when a prefix should end and a fact begins in unstructured text.

To get started with spaCy, here is a simple code snippet:

```
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp(my_text)
for token in doc:
  print(token.text, token.lower, token.pos_)
```

Additionally, spaCy will segment paragraphs into sentences: `doc.sents`. It provides lists of entities: `doc.ents` and `doc.sents[i].ents`.

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")
print("spaCy loaded.")

These are the most likely parts of speech you will find tokens associated with facts in. You may change them if necessary.


In [ ]:
### You may change this if necessary.
FACT_POS_TAGS = {"NOUN", "PROPN", "NUM"}

### A.2.b. Build Router Dataset (9 points)

The goal is to create an annotated dataset that associates a prefix (text) with a label (0=no fact next, 1=fact expected next)

To do this, you should scan each sentence in the SimpleWiki data for each person. As you scan a sentence you should make note of whether the words are part of the WikiData entry for that person and take stock of the words that led up to it.

For example: "Lillian Gordy Carter was born in Richland" should return the following records:

- `{"text": "Lillian", "label": 0}`
- `{"text": "Lillian Gordy", "label": 0}`
- `{"text": "Lillian Gordy Carter", "label": 0}`
- `{"text": "Lillian Gordy Carter was", "label": 0}`
- `{"text": "Lillian Gordy Carter was born", "label": 0}`
- `{"text": "Lillian Gordy Carter was born in", "label": 1}`
- `{"text": "Lillian Gordy Carter was born in Richland", "label": 0}`

`make_all_records_for_split(...)` will iterate through all the database entries and build the data for each individual person.

**Your task:** Implement `make_records_for_person(...)`, which will take three parameters:
 - `name`: the name of the individual to build the training records for.
 - `db_split` will be one of `db_clean_train`, `db_clean_dev`, or `db_clean_test`.
 - `title_to_text` maps names to biography text and was created in A.1.b.

You may make your own helper functions and create additional code cells as necessary. For example, to create a prefix, you may want to write a function that tells you when you first encounter a word that is in the facts about the person.

**Hint 1:** As you build the dataset for the router, you may find it advantageous to also make note of which WikiData relation is associated with the fact, the value of the fact, and the name of the person that this fact is for. These will not be used in training the router, but it will be helpful for the memory network. You can save this information in the records under the keys `relation`, `value`, and `name` for use in B.2 (In Part B of this assignment). For example: `[{"text": "this is text", "label": 0, ...}]`, where `...` represents any additional key/value pairs.

**Hint 2:** You will find that dates in SimpleWiki and WikiData use different formatting standards. SpaCy entity extraction can be used to identify date spans.

**Hint 3:** The text you want your router to function on isn't always going to look like SimpleWiki text, which uses a somewhat consistent formatting of paragraphs. You can augment the SimpleWiki data with other data from other sources or synthetically produced text samples that rephrase natural language containing facts in different ways. You can also replace entity names with special place-holder `[NAME]` tokens to keep the router from memorizing triggers related to specific names (which will also reduce the memory network's vocabulary).


You will only need to use these relations. Anything that isn't one of these relation types should not be considered a fact.

In [ ]:
SELECTED_RELATIONS = [
 'country of citizenship',
 'sex or gender',
 'occupation',
 'position held',
 'date of birth',
 'place of birth',
 'member of political party',
 'date of death',
 'educated at',
 'place of death',
 'residence',
 'spouse',
 'member of',
 'cause of death',
 'field of work',
]

In [ ]:
def make_records_for_person(name: str, db_split: Dict[str, Dict[str, str]], title_to_text: Dict[str, str]) -> List[Dict[str, Any]]:
  data_records = [] # All the data records
  ### START CODE HERE ###









  ### END CODE HERE ###
  return data_records

In [ ]:
def text_processing_function(text: str) -> str:
  """
  Optional function: If you choose to do any changes to the text before adding it into data_records,
  you can use this function (e.g., replace entity names with special place-holder
  [NAME] tokens to keep the router from memorizing triggers related to specific names).
  Otherwise, you can just return the text as is.
  """
  ### START CODE HERE ###









  ### END CODE HERE ###
  return text

This will generate the data for each of the training, development, and testing splits.

In [ ]:
def make_all_records_for_split(db_split, title_to_text):
  records = []
  for name in tqdm(sorted(db_split), desc="Annotating people"):
    records.extend(make_records_for_person(name, db_split, title_to_text))
  return records

### A.2.c. Build and Save the Annotation Datasets (0 points)

Set `FORCE_REBUILD_ANNOTATIONS` to `True` to generate the training, development, and testing data. This will also save the data to file.
Set it to `False` to load the data from file.

In [ ]:
ANNOTATION_OUTPUT_DIR = PROJECT_ROOT / "annotation"
ANNOTATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATION_TRAIN_PATH = ANNOTATION_OUTPUT_DIR / "annotation_train_records.json"
ANNOTATION_DEV_PATH = ANNOTATION_OUTPUT_DIR / "annotation_dev_records.json"
ANNOTATION_TEST_PATH = ANNOTATION_OUTPUT_DIR / "annotation_test_records.json"

In [ ]:
def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(obj: Any, path: Path):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
FORCE_REBUILD_ANNOTATIONS = False

if (
    (not FORCE_REBUILD_ANNOTATIONS)
    and ANNOTATION_TRAIN_PATH.exists()
    and ANNOTATION_DEV_PATH.exists()
    and ANNOTATION_TEST_PATH.exists()
):
    annotation_train_records = read_json(ANNOTATION_TRAIN_PATH)
    annotation_dev_records = read_json(ANNOTATION_DEV_PATH)
    annotation_test_records = read_json(ANNOTATION_TEST_PATH)
else:
    annotation_train_records = make_all_records_for_split(
        db_clean_train,
        title_to_text,
    )
    annotation_dev_records = make_all_records_for_split(
        db_clean_dev,
        title_to_text,
    )
    annotation_test_records = make_all_records_for_split(
        db_clean_test,
        title_to_text,
    )

    write_json(annotation_train_records, ANNOTATION_TRAIN_PATH)
    write_json(annotation_dev_records, ANNOTATION_DEV_PATH)
    write_json(annotation_test_records, ANNOTATION_TEST_PATH)

annotation_all_records = annotation_train_records + annotation_dev_records + annotation_test_records
print({
    "train_records": len(annotation_train_records),
    "dev_records": len(annotation_dev_records),
    "test_records": len(annotation_test_records),
})

## A.3. Train the Router (14 points)

Train a binary sequence classifier (DistilBERT) on the saved annotation data. You will:

- Implement `tokenize_record` to tokenize examples — **5 points**
- Implement `train_router` to train and select the best checkpoint — **9 points**


### A.3.a. Download a Base Model (0 points)


The most straight-forward way of fine-tuning a BERT-based model is to use the [HuggingFace transformers library](https://huggingface.co/docs/transformers/en/index). There is a [tutorial](https://huggingface.co/docs/transformers/en/training).


We recommend fine-tuning a version of the BERT language model to be this classifier. The advantage of fine-tuning BERT is that the BERT model already has a base level of language understanding from training generally on a large amount of textual data. It however is not itself a classifier.

The BERT model is a Transformer that performs sequence-to-sequence transformation of text. It was designed for text translation using positional masking. BERT also includes a *classification head*, which is used when a BERT model is fine-tuned to produce a numerical value. Should you choose to fine-tune a BERT model, you will be using this classifier head.

The model we recommend is `"distilbert/distilbert-base-uncased"` initialized through `AutoModelForSequenceClassification.from_pretrained()` with the appropriate number of output labels, which will automatically put a sigmoid (num labels = 1) or softmax (num labels > 1) on top of the BERT classification head.

The models available in the transformers package come with their own tokenizers and data collators using `AutoTokenizer.from_pretrained()`.

You will also want to be able to map back and forth between labels:
```
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}
```

However, you may choose to make your own class wrapper around a pre-trained BERT and add additional layers such as a trainable linear layers and sigmoids. If you take this route, you will either need to have your custom class `forward()` compatible with the `Trainer` object, or write your own training loop.

In [ ]:
from datetime import datetime
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

In [ ]:
# The classifier predicts two labels: NEGATIVE (0) and POSITIVE (1).
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

In [ ]:
# Load the tokenizer and a DistilBERT sequence classifier.
router_tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased", use_fast=True)
router_tokenizer.truncation_side = "left" # If your text is too long, keep the last k
router_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)
router_model.to(DEVICE)

### A.3.b. Preprocess Data (5 points)

Write `preprocess_function()` to pre-process all of your annotation records with the tokenizer. Your training, dev, and testing data should be a list of dictionaries with `text` and `label` slots.

This function should produce a list of dictionaries with the following slots:
- `input_ids`: a list of tokens from the base model tokenizer.
- `token_type_ids`: a list of zeros the same length as the input_ids.
- `attention_mask`: a list of ones the same length as the input_ids.
- `labels`: a 0 or a 1

**Note:** The base model tokenizer can automatically create a dictionary with the first three slots. You must add the fourth slot, which is `labels` plural.



In [ ]:
def tokenize_record(example: Dict[str, Any], tokenizer: AutoTokenizer, max_length: int) -> Dict[str, Any]:
    """
    Tokenize one annotation record and attach its gold label.

    Arguments:
    example -- python dictionary containing:
                    text -- prefix text, string
                    label -- binary label, integer
    tokenizer -- tokenizer to use for the model
    max_length -- maximum length of the tokenized text

    Returns:
    tokenized_output -- tokenized example with a labels field

    Note:
    Trainer expects the gold label to be stored under "labels".
    """
    ### START CODE HERE ###









    ### END CODE HERE ###
    return tokenized_output

In [ ]:
MAX_LENGTH = 16
tokenized_train = [tokenize_record(s, router_tokenizer, MAX_LENGTH) for s in tqdm(annotation_train_records, desc="Tokenizing train")]
tokenized_dev = [tokenize_record(s, router_tokenizer, MAX_LENGTH)for s in tqdm(annotation_dev_records, desc="Tokenizing dev")]
tokenized_test = [tokenize_record(s, router_tokenizer, MAX_LENGTH) for s in tqdm(annotation_test_records, desc="Tokenizing test")]

print(
    f"Tokenized records: train={len(tokenized_train)}, dev={len(tokenized_dev)}, test={len(tokenized_test)}"
)

### A.3.c. Compute Metrics (0 points)

The `compute_metrics(prediction)` takes a prediction, which will be a tuple containing `(logits, labels)` and executes `sklearn.metrics.accuracy_score` and `sklearn.metrics.f1_score` and returns a dictionary containing `"accuracy"` and `"f1"` slots.

In [ ]:
def compute_metrics(eval_pred):
    """
    Compute evaluation metrics from model logits and gold labels.

    Arguments:
    eval_pred -- tuple of the form (logits, labels)

    Returns:
    metrics -- python dictionary containing accuracy and binary f1
    """
    logits, labels = eval_pred
    # Convert logits into predicted class ids.
    preds = np.argmax(logits, axis=-1)
    metrics = {"accuracy": accuracy_score(labels, preds),
               "f1": f1_score(labels, preds, average="binary", pos_label=1, zero_division=0),
              }
    return metrics


### A.3.d. Train the Router (9 points)

Create a `TrainingArguments` object.

We recommend setting `metric_for_best_model="eval_f1"` so that the trainer will use f1 on the dev split to select the best checkpoint because this task is imbalanced, so accuracy alone is not enough. Also set `greater_is_better=True` to correspond.

Other parameters to consider setting include:
- learning_rate
- per_device_train_batch_size
- per_device_eval_batch_size
- num_train_epochs
- weight_decay
- seed
- logging_strategy="epoch"
- save_strategy="epoch"
- eval_strategy="epoch"
- load_best_model_at_end=True

Here are some suggested settings to get you started. Feel free to adjust them:

- learning_rate = 5e-6
- per_device_train_batch_size = 1024
- per_device_eval_batch_size = 1024
- num_train_epochs = 5
- weight_decay = 0.01

You can use `DataCollatorWithPadding` to create padded batches.

The `Trainer` object can now be created with the base model, training args object, training dataset, development dataset, data collators, and `compute_metrics()` function. DO NOT pass in the evaluation dataset. Use the development dataset as the `eval_dataset` only.

Create a trainer object called `trainer`. The training is then launched with: `trainer.train()`.

In [ ]:
#!pip install -q huggingface_hub

In [ ]:
HF_HANDLE = None ### Add your handle here if you want to store your models in HuggingFace Hub
if HF_HANDLE is not None:
  from huggingface_hub import notebook_login
  notebook_login()


In [ ]:
# Save each run to a folder in Google Drive.
CLASSIFIER_OUTPUT_DIR = Path(PROJECT_ROOT) / "models" / f"router"
CLASSIFIER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Override the following with your own trainer object:

In [ ]:
def train_router(model, tokenizer, train_tokenized, dev_tokenized, output_dir):
  data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

  # Evaluate once per epoch and keep the best checkpoint based on dev F1.
  training_args = TrainingArguments(
    ### START CODE HERE ###









    ### END CODE HERE ###
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_tokenized,
      eval_dataset=dev_tokenized,
      data_collator=data_collator,
      compute_metrics=compute_metrics,
  )

  # Train the classifier.
  trainer.train()

  # Save the final model and tokenizer so they can be reused later.
  trainer.save_model(str(output_dir))
  tokenizer.save_pretrained(str(output_dir))
  return trainer.model

Set `TRAIN_ROUTER` to `True` to run the trainer and train the model and save it. Set it to `False` to instead load an already-trained model.

In [ ]:
TRAIN_ROUTER = True

if TRAIN_ROUTER:
  router_model = train_router(router_model, router_tokenizer, tokenized_train, tokenized_dev, CLASSIFIER_OUTPUT_DIR)

  print("Training finished.")
  print(f"Trained Model and tokenizer saved at: {CLASSIFIER_OUTPUT_DIR}")

elif HF_HANDLE is not None:
  model_id = "{handle}/router".format(handle=HF_HANDLE)
  router_model = AutoModelForSequenceClassification.from_pretrained(model_id).to(DEVICE)
else:
  ### Load a trained model from a local file
  router_model = AutoModelForSequenceClassification.from_pretrained(str(CLASSIFIER_OUTPUT_DIR)).to(DEVICE)


## A.4. Router Evaluation (0 points)

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
#Prepare test data
data_collator = DataCollatorWithPadding(tokenizer=router_tokenizer)
test_ds = data_collator(tokenized_test).to(DEVICE)
test_labels = test_ds.pop("labels")

In [ ]:
# Predict on the test split and convert positive-class probabilities into binary labels.
# Direct model evaluation without relying on the trainer object, can work if models on loaded models too.
router_model.eval()
with torch.no_grad():
    outputs = router_model(**test_ds)
    test_logits = outputs.logits

y_true_test = np.array([int(record) for record in test_labels])
probs = F.softmax(test_logits, dim=-1).cpu().numpy()
y_pred_test = np.argmax(probs, axis=1)

print("accuracy", np.count_nonzero(y_pred_test == y_true_test) / len(y_true_test))
print(f"Test classification report:")
print(classification_report(y_true_test, y_pred_test, target_names=["NEGATIVE", "POSITIVE"], zero_division=0))

### Student Self-check/local autograder:

Put `test_part_A.py` in your working directory and run the cell below. These checks match the autograder.


In [ ]:
from test_part_A import run_all
# Data labeling run onsynthetic and real people sample.
run_all(make_records_for_person, text_processing_function, title_to_text, db_clean_all, tokenize_record, train_router)

Submit your completed `.ipynb` file to **HW5 Part A** on Gradescope.
